# Load Dataset

In [ ]:
!pip install datasets --q

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, Features, Image as HFImage, ClassLabel
from huggingface_hub import hf_hub_download
import zipfile
import os
import tempfile

In [ ]:
def load_liveness_dataset():
    """Load liveness dataset. Uses fallback (download zip, build from filenames) if HF loader fails due to invalid labels."""
    try:
        return load_dataset("Bahareh0281/liveness_images")
    except Exception as e:
        if "Invalid string class label" not in str(e) and "DatasetGenerationError" not in type(e).__name__:
            raise
        print("Dataset loader failed (invalid label in repo). Loading from zip with filename-based labels (_real=1, _fake=0)...")
    data_dir = tempfile.mkdtemp(prefix="liveness_")
    zip_path = hf_hub_download(repo_id="Bahareh0281/liveness_images", filename="liveness_images.zip", repo_type="dataset")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)
    paths, labels = [], []
    for root, _, files in os.walk(data_dir):
        for f in files:
            if not f.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            path = os.path.join(root, f)
            rel_parts = os.path.relpath(root, data_dir).split(os.sep)
            rel_parts_lower = [p.lower() for p in rel_parts]
            f_lower = f.lower()
            label = None
            if "fake" in rel_parts_lower or "fake" in f_lower:
                label = 0
            elif "real" in rel_parts_lower or "real" in f_lower:
                label = 1
            elif any(p == "0" for p in rel_parts_lower):
                label = 0
            elif any(p == "1" for p in rel_parts_lower):
                label = 1
            if label is not None:
                paths.append(path)
                labels.append(label)
    if not paths:
        seen = []
        for root, _, files in os.walk(data_dir):
            imgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            if imgs:
                rel = os.path.relpath(root, data_dir)
                seen.append((rel, imgs[:3]))
            if len(seen) >= 5:
                break
        msg = "No images with 'real'/'fake' in path or filename found in zip."
        if seen:
            msg += " Sample paths: " + str(seen[:5])
        raise RuntimeError(msg)
    ds = Dataset.from_dict({"image": paths, "label": labels})
    ds = ds.cast_column("image", HFImage()).cast_column("label", ClassLabel(num_classes=2, names=["fake", "real"]))
    return DatasetDict({"train": ds})

Dataset = load_liveness_dataset()

In [ ]:
Dataset['train'][7]

# Import Necessary Libraries

In [ ]:
import cv2
import numpy as np
import os
from skimage.feature import local_binary_pattern
from skimage import measure
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Feature Extraction Functions

In [ ]:
radius = 3
n_points = 8 * radius

def compute_fourier_transform(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift))
    return magnitude_spectrum

def compute_lbp(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
    return lbp

def compute_depth(image):
    depth = image[:, :, 2]
    return depth

def extract_statistical_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mean = np.mean(gray)
    std_dev = np.std(gray)
    skewness = np.mean((gray - mean) ** 3) / (std_dev ** 3)
    kurtosis = np.mean((gray - mean) ** 4) / (std_dev ** 4)
    entropy = measure.shannon_entropy(gray)
    return mean, std_dev, skewness, kurtosis, entropy

# Preprocess Input Images

In [ ]:
len(Dataset['train'])

In [ ]:
def process_images(dataset, traget_size, num=0):
    train_images_features = []
    train_images_labels = []
    if num == 0:
        num = len(dataset['train'])

    for i in range(num):
        img = dataset['train'][i]['image']
        if isinstance(img, Image.Image):
            img = np.array(img)  # Convert PIL image to NumPy array

            # Extract frequency features
            magnitude_spectrum = compute_fourier_transform(img)
            magnitude_spectrum_resized = cv2.resize(magnitude_spectrum, (traget_size, traget_size))

            # Extract LBP features
            lbp = compute_lbp(img)
            lbp_hist, _ = np.histogram(lbp, bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
            lbp_hist_normalized = lbp_hist / lbp_hist.sum()
            lbp_hist_resized = cv2.resize(lbp_hist_normalized.reshape(-1, 1), (traget_size, traget_size))

            # Extract statistical features
            mean, std_dev, skewness, kurtosis, entropy = extract_statistical_features(img)
            statistical_features = np.array([mean, std_dev, skewness, kurtosis, entropy])
            statistical_features_resized = cv2.resize(statistical_features.reshape(-1, 1), (traget_size, traget_size))

            # Combine features into a 3D array
            combined_features = np.stack([
                magnitude_spectrum_resized,
                lbp_hist_resized,
                statistical_features_resized
            ], axis=-1)

            train_images_features.append(combined_features)
            train_images_labels.append(dataset['train'][i]['label'])

    return np.array(train_images_features), np.array(train_images_labels)

In [43]:
train_images_features, train_images_labels = process_images(Dataset, 64, 3000)

C:\Users\Ominisoft\AppData\Local\Temp\ipykernel_4272\284126899.py:8: RuntimeWarning: divide by zero encountered in log
  magnitude_spectrum = 20 * np.log(np.abs(fshift))


In [44]:
len(train_images_features)

3000

In [45]:
train_images_features.shape

(3000, 64, 64, 3)

In [46]:
train_images_features[1]

array([[[9.98781304e+01, 1.70135498e-02, 1.23999863e+02],
        [1.03112881e+02, 1.70135498e-02, 1.23999863e+02],
        [1.00401371e+02, 1.70135498e-02, 1.23999863e+02],
        ...,
        [8.08424041e+01, 1.70135498e-02, 1.23999863e+02],
        [8.49438029e+01, 1.70135498e-02, 1.23999863e+02],
        [9.55299601e+01, 1.70135498e-02, 1.23999863e+02]],

       [[7.85450649e+01, 1.65896416e-02, 1.23999863e+02],
        [8.88281131e+01, 1.65896416e-02, 1.23999863e+02],
        [8.47440767e+01, 1.65896416e-02, 1.23999863e+02],
        ...,
        [9.16315104e+01, 1.65896416e-02, 1.23999863e+02],
        [9.06426121e+01, 1.65896416e-02, 1.23999863e+02],
        [8.25498088e+01, 1.65896416e-02, 1.23999863e+02]],

       [[9.15188864e+01, 1.50151253e-02, 1.23999863e+02],
        [9.45883978e+01, 1.50151253e-02, 1.23999863e+02],
        [9.23813503e+01, 1.50151253e-02, 1.23999863e+02],
        ...,
        [9.37271837e+01, 1.50151253e-02, 1.23999863e+02],
        [9.23618173e+01, 1.50

In [47]:
len(train_images_features[1])

64

In [48]:
len(train_images_features[0])

64

# Split training dataset and prepare it for train process

In [49]:
# Convert depth features to a numpy array
features = np.array(train_images_features)
labels = np.array(train_images_labels)
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# One-hot encode the labels
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)


# Create CNN Model and Train it

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)


Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.7484 - loss: 8.3251 - val_accuracy: 0.8271 - val_loss: 0.6005
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.7948 - loss: 0.5669 - val_accuracy: 0.8271 - val_loss: 0.5004
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.7948 - loss: 0.5205 - val_accuracy: 0.8271 - val_loss: 0.4708
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7948 - loss: 0.5111 - val_accuracy: 0.8271 - val_loss: 0.4652
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7948 - loss: 0.5118 - val_accuracy: 0.8271 - val_loss: 0.4648
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7948 - loss: 0.5133 - val_accuracy: 0.8271 - val_loss: 0.4647
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.7948 - loss: 0.5153 - val_accuracy: 0.8271 - val_loss: 0.4640
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.7948 - loss: 0.5117 - val_accuracy: 0.8271 - v

# Evaluate on dataset tests

In [51]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8100 - loss: 0.4877
Test Accuracy: 81.00%


# Load test videos

In [52]:
import gdown

file_id = '1a5R5h05hCyw9PzIBhSjy2jLL3dSFy2xA'
import os
data_dir = os.path.join('.', 'data')
os.makedirs(data_dir, exist_ok=True)
destination = os.path.join(data_dir, 'dataset.zip')  # Windows-compatible path
gdown.download(f'https://drive.google.com/uc?id={file_id}', destination, quiet=False)

import zipfile

with zipfile.ZipFile(destination, 'r') as zip_ref:
    extract_dir = os.path.join(data_dir, 'dataset')
    os.makedirs(extract_dir, exist_ok=True)
    zip_ref.extractall(extract_dir)


Downloading...
From (original): https://drive.google.com/uc?id=1a5R5h05hCyw9PzIBhSjy2jLL3dSFy2xA
From (redirected): https://drive.google.com/uc?id=1a5R5h05hCyw9PzIBhSjy2jLL3dSFy2xA&confirm=t&uuid=a7fc0068-60f6-4576-9c89-62add76d0f6a
To: D:\Projects\Face-Modle\Video-Based-Face-Liveness-Detection\notebooks\data\dataset.zip
100%|█████████████████████████████████████████████| 377M/377M [00:30<00:00, 12.5MB/s]


# Generate random frames from each video

In [53]:
import random

def extract_frames(video_path, save_path, label, test):
    # Open the video file
    video = cv2.VideoCapture(video_path)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    # Select one random frame
    random_frame = random.randint(0, frame_count - 1)

    # Set the position of the video to the selected frame
    video.set(cv2.CAP_PROP_POS_FRAMES, random_frame)
    success, frame = video.read()

    # If the frame was successfully read, save it
    if success:
        frame_path = os.path.join(save_path, f"{label}_{random_frame}.jpg")
        cv2.imwrite(frame_path, frame)

        # Convert the frame to a PIL image
        pil_image = Image.open(frame_path)

        # Save the image and label to the dictionary
        test.append({'image': pil_image, 'label': label})

    # Release the video file
    video.release()

In [54]:
import cv2

import os
data_dir = os.path.join('.', 'data')
fake_test_videos_path = os.path.join(data_dir, 'dataset', 'fake', 'test')
real_test_videos_path = os.path.join(data_dir, 'dataset', 'real', 'test')

save_frames_path = os.path.join(data_dir, 'extracted_frames', 'test')
# Create the directory if it doesn't exist
if not os.path.exists(save_frames_path):
    os.makedirs(save_frames_path)

# Create a list to hold the dictionary entries
test = []

# Iterate over fake videos and extract frames
for fake_video_file in os.listdir(fake_test_videos_path):
    fake_video_path = os.path.join(fake_test_videos_path, fake_video_file)
    extract_frames(fake_video_path, save_frames_path, 0, test)

# Iterate over real videos and extract frames
for real_video_file in os.listdir(real_test_videos_path):
    real_video_path = os.path.join(real_test_videos_path, real_video_file)
    extract_frames(real_video_path, save_frames_path, 1, test)

In [55]:
len(test)

42

# Extract features from each frame

In [56]:
from PIL import Image
import numpy as np
import cv2
from skimage.feature import local_binary_pattern

# Function to compute Fourier Transform (dummy implementation)
def compute_fourier_transform(image):
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift))
    return magnitude_spectrum

# Function to compute LBP (Local Binary Pattern) (dummy implementation)
def compute_lbp(image, P=8, R=1):
    lbp = local_binary_pattern(image, P, R, method="uniform")
    return lbp

# Function to extract statistical features (dummy implementation)
def extract_statistical_features(image):
    mean = np.mean(image)
    std_dev = np.std(image)
    skewness = np.mean((image - mean)**3) / (std_dev**3)
    kurtosis = np.mean((image - mean)**4) / (std_dev**4) - 3
    entropy = -np.sum(image * np.log2(image + 1e-9))
    return mean, std_dev, skewness, kurtosis, entropy

def process_images_2(tests, target_size):
    test_images_features = []
    test_images_labels = []

    for i in range(len(tests)):
        item = tests[i]

        # Debugging: print the item and its keys
        print(f"Item {i}: {item}")
        print(f"Keys: {item.keys()}")

        if 'image' not in item or 'label' not in item:
            raise KeyError(f"Item at index {i} must contain 'image' and 'label' keys.")

        img = item['image']
        if isinstance(img, Image.Image):
            img = np.array(img)  # Convert PIL image to NumPy array
        elif isinstance(img, np.ndarray):
            pass  # It's already a NumPy array
        else:
            raise TypeError(f"Item at index {i} is neither a PIL image nor a NumPy array.")

        # Convert to grayscale if necessary
        if img.ndim == 3 and img.shape[-1] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # Extract frequency features
        magnitude_spectrum = compute_fourier_transform(img)
        magnitude_spectrum_resized = cv2.resize(magnitude_spectrum, (target_size, target_size))

        # Extract LBP features
        lbp = compute_lbp(img)
        lbp_hist, _ = np.histogram(lbp, bins=np.arange(0, 8 + 3), range=(0, 8 + 2))
        lbp_hist_normalized = lbp_hist / lbp_hist.sum()
        lbp_hist_resized = cv2.resize(lbp_hist_normalized.reshape(-1, 1), (target_size, target_size))

        # Extract statistical features
        mean, std_dev, skewness, kurtosis, entropy = extract_statistical_features(img)
        statistical_features = np.array([mean, std_dev, skewness, kurtosis, entropy])
        statistical_features_resized = cv2.resize(statistical_features.reshape(-1, 1), (target_size, target_size))

        # Combine features into a 3D array
        combined_features = np.stack([
            magnitude_spectrum_resized,
            lbp_hist_resized,
            statistical_features_resized
        ], axis=-1)

        test_images_features.append(combined_features)
        test_images_labels.append(item['label'])

    return np.array(test_images_features), np.array(test_images_labels)

# Extract features from each frame and convert labels to one-hot form

In [57]:
test_frames_features, test_frames_labels = process_images_2(test, 64)
test_frames_labels = to_categorical(test_frames_labels, num_classes=2)

Item 0: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1080x2400 at 0x22060107450>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 1: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720 at 0x2205FC97850>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 2: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x1280 at 0x2205FF18610>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 3: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x928 at 0x21FF85104D0>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 4: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x992 at 0x2205897D310>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 5: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x816 at 0x22003EBA890>, 'label': 0}
Keys: dict_keys(['image', 'label'])
Item 6: {'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720 at 0x2206040D650>, 'label'

In [58]:
test_frames_features.shape

(42, 64, 64, 3)

In [59]:
test_frames_labels.shape

(42, 2)

# Evaluate model on test set

In [60]:
test_loss, test_accuracy = model.evaluate(test_frames_features, test_frames_labels)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.5476 - loss: 15683829.0000
Test Accuracy: 54.76%


In [61]:
import numpy as np

# Apply softmax to get normalized probabilities
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

predictions = model.predict(test_frames_features)

normalized_predictions = softmax(predictions)
class_1_normalized_predictions = normalized_predictions[:, 1]

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step


In [62]:
class_1_normalized_predictions

array([0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414,
       0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414, 0.2689414],
      dtype=float32)

In [63]:
y_pred_proba = model.predict(test_frames_features)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


In [64]:
test_frames_features.shape

(42, 64, 64, 3)

In [65]:
def load_image_paths(directory):
    image_paths = []
    for filename in os.listdir(directory):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            image_paths.append(os.path.join(directory, filename))
    return image_paths

In [66]:
import os
import os
data_dir = os.path.join('.', 'data')
test_image_pathss = load_image_paths(os.path.join(data_dir, 'extracted_frames', 'test'))

In [67]:
test_image_pathss

['.\\data\\extracted_frames\\test\\0_0.jpg',
 '.\\data\\extracted_frames\\test\\0_10.jpg',
 '.\\data\\extracted_frames\\test\\0_11.jpg',
 '.\\data\\extracted_frames\\test\\0_12.jpg',
 '.\\data\\extracted_frames\\test\\0_125.jpg',
 '.\\data\\extracted_frames\\test\\0_129.jpg',
 '.\\data\\extracted_frames\\test\\0_13.jpg',
 '.\\data\\extracted_frames\\test\\0_135.jpg',
 '.\\data\\extracted_frames\\test\\0_14.jpg',
 '.\\data\\extracted_frames\\test\\0_15.jpg',
 '.\\data\\extracted_frames\\test\\0_152.jpg',
 '.\\data\\extracted_frames\\test\\0_156.jpg',
 '.\\data\\extracted_frames\\test\\0_159.jpg',
 '.\\data\\extracted_frames\\test\\0_16.jpg',
 '.\\data\\extracted_frames\\test\\0_172.jpg',
 '.\\data\\extracted_frames\\test\\0_175.jpg',
 '.\\data\\extracted_frames\\test\\0_19.jpg',
 '.\\data\\extracted_frames\\test\\0_20.jpg',
 '.\\data\\extracted_frames\\test\\0_22.jpg',
 '.\\data\\extracted_frames\\test\\0_23.jpg',
 '.\\data\\extracted_frames\\test\\0_240.jpg',
 '.\\data\\extracted_frame

In [68]:
import csv
# Save predictions to a CSV file
output_file = 'predictions.csv'
with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'liveness_score_frequency'])  # Header
    for i, filename in enumerate(test_image_pathss):
        writer.writerow([filename, class_1_normalized_predictions[i]])

print(f"Predictions saved to {output_file}")

IndexError: index 42 is out of bounds for axis 0 with size 42

# InceptionV3


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import img_to_array, array_to_img
from tensorflow.keras.preprocessing.image import load_img

In [ ]:
train_images_features, train_images_labels = process_images(Dataset, 75, 3000)

In [ ]:
# Convert depth features to a numpy array
features2 = np.array(train_images_features)
labels2 = np.array(train_images_labels)
# Split the data into training and testing sets
X_train2, X_test2, y_train2, y_test2 = train_test_split(features2, labels2, test_size=0.2, random_state=42)

# One-hot encode the labels
y_train2 = to_categorical(y_train2, num_classes=2)
y_test2 = to_categorical(y_test2, num_classes=2)

In [ ]:
# Load the pre-trained model
inception_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(75, 75, 3))  # Exclude the top layer

# Freeze the layers of the pre-trained model
for layer in inception_model.layers:
    layer.trainable = False

# Add a global spatial average pooling layer
x = inception_model.output
x = GlobalAveragePooling2D()(x)

# Add a fully-connected layer
x = Dense(1024, activation='relu')(x)

# Add a logistic layer with the number of classes you have (binary classification)
predictions = Dense(2, activation='softmax')(x)

# This is the model we will train
model2 = Model(inputs=inception_model.input, outputs=predictions)

In [ ]:
# Compile the model
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model2.fit(X_train2, y_train2, epochs=10, batch_size=32, validation_data=(X_test2, y_test2))

In [ ]:
# unfreeze the layers of the pre-trained model
for layer in model2.layers:
    layer.trainable = True

In [ ]:
# Compile the model
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model2.fit(X_train2, y_train2, epochs=10, batch_size=32, validation_data=(X_test2, y_test2))

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model2.evaluate(X_test2, y_test2)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

In [ ]:
test_frames_features2, test_frames_labels2 = process_images_2(test, 75)
test_frames_labels2 = to_categorical(test_frames_labels2, num_classes=2)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_frames_features2, test_frames_labels2)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

In [ ]:
# Make predictions

y_pred_proba = model2.predict(test_frames_features2)
predictions_for_frames = []
# Output the prediction vector for each test image
for idx, prediction_vector in enumerate(y_pred_proba):
    print(f"Prediction vector for test image {idx+1}: {prediction_vector}")
    predictions_for_frames.append((idx, prediction_vector))


# Crop

In [ ]:
!pip install mtcnn --q

In [ ]:
from mtcnn import MTCNN

In [ ]:
import os
import os
data_dir = os.path.join('.', 'data')
test_images_dir = os.path.join(data_dir, 'extracted_frames2', 'test')

In [ ]:
def load_image_paths(directory):
    image_paths = []
    for filename in os.listdir(directory):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            image_paths.append(os.path.join(directory, filename))
    return image_paths

In [ ]:
def detect_and_crop_faces(image_path, detector):
    image = Image.open(image_path)
    image_np = np.asarray(image)
    result = detector.detect_faces(image_np)
    if result:
        for person in result:
            bounding_box = person['box']
            keypoints = person['keypoints']

            # Crop the detected face
            x, y, width, height = bounding_box
            cropped_face = image_np[y:y+height, x:x+width]

            # Convert the cropped face back to an image
            cropped_face_image = Image.fromarray(cropped_face)

            return cropped_face_image
    return None

In [ ]:
# Initialize the MTCNN face detector
detector = MTCNN()

In [ ]:
import os
import os
data_dir = os.path.join('.', 'data')
# Directory containing the original images
test_image_paths = load_image_paths(os.path.join(data_dir, 'extracted_frames', 'test'))

# Directory to save cropped face images
output_dir = os.path.join(data_dir, 'cropped_faces')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Process each image and save the cropped face with the original filename
for image_path in test_image_paths:
    cropped_face = detect_and_crop_faces(image_path, detector)
    # print("Image path is:", test_image_paths)
    if cropped_face:
        # Extract the original filename
        original_filename = os.path.basename(image_path)
        # Save the cropped face image with the original filename
        cropped_face.save(os.path.join(output_dir, original_filename))

In [ ]:
# Function to resize images in a directory
def resize_images(directory, target_size=(64, 64)):
    resized_images = []
    test_image_paths = []
    test_image_labels = []

    # Iterate over each image in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            label = filename.split('.')[0].split('_')[0]
            image_path = os.path.join(directory, filename)
            image = cv2.imread(image_path)
            resized_image = cv2.resize(image, target_size)

            # Append resized image, image path, and label
            resized_images.append(resized_image)
            test_image_paths.append(image_path)
            test_image_labels.append(int(label))  # Convert label to int

    # Convert lists to numpy arrays
    resized_images = np.array(resized_images)
    test_image_labels = np.array(test_image_labels)

    return resized_images, test_image_paths, test_image_labels

In [ ]:
# Resize images in test directory
cropped_resized_images, cropped_test_image_paths, cropped_test_image_labels = resize_images(output_dir)

# Check the shape of resized images
print("Resized images shape:", cropped_resized_images.shape)

In [ ]:
# Convert labels to one-hot encoding
cropped_test_image_labels_onehot = to_categorical(cropped_test_image_labels, num_classes=2)

In [ ]:
# Function to process images
def process_images_2(images, target_size):
    processed_images = []
    for image in images:
        # Resize the image
        resized_image = cv2.resize(image, (target_size, target_size))
        # Normalize pixel values to range [0, 1]
        normalized_image = resized_image / 255.0
        processed_images.append(normalized_image)

    return np.array(processed_images)

# Resize images in test directory
cropped_resized_images, cropped_test_image_paths, cropped_test_image_labels

In [ ]:
# Evaluate the model
cropped_test_loss, cropped_test_accuracy = model.evaluate(cropped_resized_images, cropped_test_image_labels_onehot)

In [ ]:
# Predictions
cropped_predictions = model.predict(cropped_resized_images)
cropped_predicted_scores = cropped_predictions[:, 1]  # Assuming class 1 corresponds to index 1 in predictions

# Normalize scores between 0 and 1
cropped_predicted_scores_normalized = (cropped_predicted_scores - np.min(cropped_predicted_scores)) / (np.max(cropped_predicted_scores) - np.min(cropped_predicted_scores))

In [ ]:
# Save predictions to a CSV file
output_file = 'predictions.csv'
with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'liveness_score_Frequency', 'liveness_score_Frequency_crop'])  # Header
    for i, filename in enumerate(test_image_paths):
        writer.writerow([filename, class_1_normalized_predictions[i], cropped_predicted_scores_normalized[i]])

print(f"Predictions saved to {output_file}")